# 1강: NLP 기초와 법률 데이터 규칙 기반 분류

이 노트북은 자연어 처리(NLP)의 핵심 개념을 법률 문장 분류 실습으로 익히는 교육용 자료입니다.

## 학습 목표
1. 토크나이제이션 — 텍스트를 토큰으로 분해하는 방법
2. 임베딩 — 텍스트를 숫자 벡터로 표현하는 방법
3. 자기회귀 언어모델 — bigram 예측으로 LLM 원리 시뮬레이션
4. 법률 조항 6개 카테고리 정의 이해
5. 규칙 기반 분류기 구현 및 한계 확인

## 전체 처리 흐름

```
법률 조항 원문  ->  토큰화  ->  수치 표현(임베딩)  ->  모델 또는 규칙  ->  카테고리 예측  ->  근거 검토
```


## 0. 라이브러리 임포트

In [ ]:
import re
import itertools
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('라이브러리 로드 완료')


## 1. 토크나이제이션

텍스트를 모델이 처리할 수 있는 작은 단위인 토큰으로 나누는 과정입니다.

토크나이저 선택에 따라 같은 문장도 다른 토큰이 되고, 그 결과 분류 성능도 달라집니다.

**공백 기준 분할** — 가장 단순한 방법. 조사가 붙은 채로 토큰이 됩니다.

**정규표현식 기준 분할** — 한글, 영문, 숫자만 추출합니다.


In [ ]:
# 1-1. 토크나이제이션 비교

legal_sentence = '법원은 피고인에게 징역 3년과 벌금 500만원을 선고했다'

space_tokens = legal_sentence.split()
regex_tokens = re.findall(r'[가-힣A-Za-z0-9]+', legal_sentence)

print('원문:', legal_sentence)
print()
print('공백 기준 토큰 ({} 개)'.format(len(space_tokens)))
print(space_tokens)
print()
print('정규표현식 토큰 ({} 개)'.format(len(regex_tokens)))
print(regex_tokens)

legal_keywords = ['법원', '피고인', '징역', '벌금', '선고']
print()
print('법률 핵심어 포함 여부')
for kw in legal_keywords:
    print('  {}: {}'.format(kw, '있음' if kw in legal_sentence else '없음'))


## 2. 임베딩 — 텍스트를 숫자 벡터로

컴퓨터는 단어 자체를 이해하지 못합니다. 숫자로 바꿔야 계산할 수 있습니다.

이 실습에서는 가장 단순한 방법인 BoW(Bag of Words)로 간이 임베딩을 만들고,
코사인 유사도로 문장 간 의미 거리를 측정합니다.

같은 카테고리 문장끼리는 유사도가 높고, 다른 카테고리 문장끼리는 낮아야 합니다.

| 방법 | 설명 |
|------|------|
| BoW | 단어 등장 여부를 0, 1로 표현 |
| Word2Vec / FastText | 주변 단어 예측으로 의미 학습 |
| Transformer 임베딩 | 문맥 전체를 반영한 동적 벡터 (GPT, BERT 등) |


In [ ]:
# 2-1. BoW 간이 임베딩 + 코사인 유사도

embedding_examples = pd.DataFrame([
    {'id': 'S1', 'label': 'PROC',  'text': '위반한 자는 3년 이하의 징역 또는 벌금에 처한다.'},
    {'id': 'S2', 'label': 'PROC',  'text': '거짓 신고를 한 자는 1천만원 이하의 벌금에 처한다.'},
    {'id': 'S3', 'label': 'ORG',   'text': '분쟁 조정을 위하여 조정위원회를 설치한다.'},
    {'id': 'S4', 'label': 'ORG',   'text': '위원회는 위원장 1명을 포함하여 15명 이내의 위원으로 구성한다.'},
    {'id': 'S5', 'label': 'RIGHT', 'text': '근로자는 안전한 환경에서 일할 권리를 가진다.'},
])

vocabulary = ['징역', '벌금', '처한다', '위원회', '설치', '구성', '권리', '근로자', '안전']

for word in vocabulary:
    embedding_examples[word] = embedding_examples['text'].str.contains(word).astype(int)

print('문장별 BoW 임베딩 벡터')
print(embedding_examples[['id', 'label'] + vocabulary].to_string())


def cosine_similarity(vec_a, vec_b):
    """두 벡터의 코사인 유사도를 반환합니다 (0~1, 높을수록 유사)."""
    a = np.array(vec_a, dtype=float)
    b = np.array(vec_b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0


rows = []
for i in range(len(embedding_examples)):
    for j in range(i + 1, len(embedding_examples)):
        ri = embedding_examples.iloc[i]
        rj = embedding_examples.iloc[j]
        sim = cosine_similarity(ri[vocabulary], rj[vocabulary])
        rows.append({
            'pair':          '{}-{}'.format(ri['id'], rj['id']),
            'labels':        '{} vs {}'.format(ri['label'], rj['label']),
            'similarity':    round(sim, 3),
            'same_category': ri['label'] == rj['label'],
        })

sim_df = pd.DataFrame(rows).sort_values('similarity', ascending=False)
print()
print('문장 쌍 코사인 유사도 (높을수록 의미적으로 가까움)')
print(sim_df.to_string(index=False))
print()
print('같은 카테고리(same_category=True) 쌍의 유사도가 더 높을수록 분류에 유용합니다.')


## 3. LLM 작동 원리 — 자기회귀 토큰 예측

LLM은 기본적으로 앞 단어들을 보고 다음 단어를 예측하는 모델입니다.

```
P(다음_토큰 | 앞의_토큰들)
```

실제 GPT나 LLaMA는 수십억 개의 파라미터로 이 확률을 계산하지만,
bigram 빈도 모델로 같은 원리를 단순하게 시뮬레이션할 수 있습니다.

| 단계 | 설명 |
|------|----- |
| 1 | 말뭉치에서 연속된 두 토큰 빈도를 셉니다 |
| 2 | 현재 토큰 뒤에 올 토큰의 확률을 추정합니다 |
| 3 | 가장 확률이 높은 토큰을 반복 선택해 문장을 생성합니다 |


In [ ]:
# 3-1. Bigram 언어모델로 LLM 원리 시뮬레이션

corpus = [
    '이 법은 국민의 권리를 보호함을 목적으로 한다',
    '이 법은 개인정보를 안전하게 보호함을 목적으로 한다',
    '위원회는 분쟁 조정을 위하여 설치한다',
    '위원회는 위원장 1명을 포함하여 구성한다',
    '위반한 자는 벌금에 처한다',
    '위반한 자는 징역에 처한다',
]


def tokenize(text):
    return re.findall(r'[가-힣A-Za-z0-9]+', text)


bigram_counts = {}
for sentence in corpus:
    tokens = ['<START>'] + tokenize(sentence) + ['<END>']
    for cur, nxt in zip(tokens, tokens[1:]):
        bigram_counts.setdefault(cur, Counter())[nxt] += 1

print('특정 토큰 뒤에 올 가능성이 높은 후보')
for tok in ['<START>', '이', '법은', '위원회는', '위반한', '자는']:
    print('  {:12s} -> {}'.format(tok, bigram_counts.get(tok, Counter()).most_common()))


def generate_by_bigram(seed, max_steps=10):
    """seed 토큰에서 시작해 bigram 최빈값으로 문장을 생성합니다."""
    current = seed
    generated = [] if seed == '<START>' else [seed]
    trace = []
    for _ in range(max_steps):
        candidates = bigram_counts.get(current, Counter())
        if not candidates:
            break
        nxt, cnt = candidates.most_common(1)[0]
        trace.append({
            'context':    ' '.join(generated) if generated else '<START>',
            'next_token': nxt,
            'frequency':  cnt,
        })
        if nxt == '<END>':
            break
        generated.append(nxt)
        current = nxt
    return ' '.join(generated), pd.DataFrame(trace)


result, trace_df = generate_by_bigram('<START>')
print()
print('생성된 문장:', result)
print()
print('생성 과정 (각 단계에서 가장 높은 빈도 토큰 선택)')
print(trace_df.to_string(index=False))
print()
print('이 단순 모델처럼 GPT도 앞 맥락을 보고 다음 토큰을 예측하는 방식으로 동작합니다.')


## 4. 법률 데이터의 분류 어려움

법률 텍스트를 자동 분류할 때 마주치는 3가지 구조적 어려움입니다.

| 어려움 | 설명 | 예시 |
|--------|------|------|
| 중첩성 | 한 문장에 여러 카테고리 신호 공존 | 청구 절차를 안내하여야 한다 -> RIGHT? PROC? |
| 맥락 의존성 | 본문만으로는 판단 불가 | 적용한다 -> DEF? ETC? (조항 제목 필요) |
| 도메인 특화성 | 법률 용어를 모르면 신호를 못 잡음 | 항소인, 송달, 상고장 |


In [ ]:
# 4-1. 법률 데이터 특성 분석

keyword_rules = {
    'DEF':   ['목적', '정의', '뜻', '말한다', '적용한다', '범위'],
    'ORG':   ['위원회', '기관', '법원', '설치', '구성', '관장', '지휘'],
    'CRIT':  ['기준', '요건', '자격', '이상', '이하', '이내', '자본금'],
    'PROC':  ['신청', '심사', '조사', '청문', '소송', '절차', '징역', '벌금', '처한다', '취소', '청구'],
    'RIGHT': ['권리', '의무', '하여야 한다', '금지', '책임', '침해', '차별'],
    'ETC':   ['시행', '공포', '경과', '부칙', '대통령령으로 정한다'],
}


def matched_categories(text, rules):
    """텍스트에 매칭된 카테고리와 해당 키워드를 반환합니다."""
    return {
        cat: [kw for kw in kws if kw in text]
        for cat, kws in rules.items()
        if any(kw in text for kw in kws)
    }


difficulty_cases = [
    {
        'case':    '중첩성',
        'context': '',
        'text':    '개인정보처리자는 정보주체의 열람 청구 절차를 지체 없이 안내하여야 한다.',
        'point':   '권리.의무(RIGHT)와 절차(PROC) 신호가 한 문장에 등장',
    },
    {
        'case':    '맥락 의존성',
        'context': '부칙',
        'text':    '제3조의 개정규정은 이 법 시행 후 최초로 접수된 사건부터 적용한다.',
        'point':   '적용한다는 DEF 신호처럼 보이지만 부칙 맥락에서는 ETC',
    },
    {
        'case':    '도메인 특화성',
        'context': '',
        'text':    '항소인은 판결서가 송달된 날부터 2주 이내에 상고장을 제출할 수 있다.',
        'point':   '항소인.판결서.송달.상고장 같은 법률 전문 용어가 핵심 단서',
    },
]

for case in difficulty_cases:
    matched = matched_categories(case['text'], keyword_rules)
    print('[{}]'.format(case['case']))
    print('  문장  :', case['text'])
    if case['context']:
        print('  맥락  :', case['context'])
    print('  포인트:', case['point'])
    print('  매칭된 카테고리:', matched)
    print()


## 5. 실습용 법률 조항 데이터셋

6개 카테고리 × 4문장 = 24건으로 구성된 교육용 데이터셋입니다.

| 코드 | 카테고리명 | 핵심 판정 기준 |
|------|-----------|---------------|
| DEF  | 정의 및 범위 | 목적, 용어 정의, 적용 범위 |
| RIGHT| 권리 및 의무 | 권리, 의무, 금지, 책임 |
| PROC | 절차 및 처벌 | 신청, 심사, 처벌, 불복 절차 |
| ORG  | 조직 및 기구 | 위원회, 기관 설치, 구성, 권한 |
| CRIT | 기준 및 요건 | 자격, 수치 기준, 요건 |
| ETC  | 기타 조항   | 시행일, 경과조치, 위임 |


In [ ]:
# 5-1. 데이터셋 구성

sample_data = [
    {'id':'D01','category':'DEF', 'text':'이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.'},
    {'id':'D02','category':'DEF', 'text':'이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.'},
    {'id':'D03','category':'DEF', 'text':'공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.'},
    {'id':'D04','category':'DEF', 'text':'이 법은 대한민국 영역 안에서 이루어지는 정보 처리 행위에 적용한다.'},
    {'id':'R01','category':'RIGHT','text':'모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 아니한다.'},
    {'id':'R02','category':'RIGHT','text':'사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.'},
    {'id':'R03','category':'RIGHT','text':'근로자는 안전하고 건강한 근무 환경에서 일할 권리를 가진다.'},
    {'id':'R04','category':'RIGHT','text':'누구든지 정당한 사유 없이 타인의 통신비밀을 침해하여서는 아니 된다.'},
    {'id':'P01','category':'PROC','text':'이 법을 위반한 자는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처한다.'},
    {'id':'P02','category':'PROC','text':'신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.'},
    {'id':'P03','category':'PROC','text':'장관은 위반 사실을 조사한 후 청문 절차를 거쳐 등록을 취소할 수 있다.'},
    {'id':'P04','category':'PROC','text':'불법행위로 인한 손해배상 청구는 민사소송법에서 정한 절차에 따른다.'},
    {'id':'O01','category':'ORG', 'text':'분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.'},
    {'id':'O02','category':'ORG', 'text':'위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.'},
    {'id':'O03','category':'ORG', 'text':'법원은 사법권을 행사하며 대법원, 고등법원 및 지방법원으로 구성된다.'},
    {'id':'O04','category':'ORG', 'text':'중앙행정기관의 장은 소관 사무를 관장하고 소속 공무원을 지휘한다.'},
    {'id':'C01','category':'CRIT','text':'후보자는 선거일 현재 25세 이상인 국민이어야 한다.'},
    {'id':'C02','category':'CRIT','text':'지원 자격은 해당 분야 경력 3년 이상 및 학사 학위 이상으로 한다.'},
    {'id':'C03','category':'CRIT','text':'안전관리 기준은 시설 면적, 이용 인원 및 위험도에 따라 대통령령으로 정한다.'},
    {'id':'C04','category':'CRIT','text':'허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.'},
    {'id':'E01','category':'ETC', 'text':'이 법은 공포 후 6개월이 경과한 날부터 시행한다.'},
    {'id':'E02','category':'ETC', 'text':'이 법 시행 당시 종전의 규정에 따라 한 처분은 이 법에 따른 처분으로 본다.'},
    {'id':'E03','category':'ETC', 'text':'이 법의 시행에 필요한 사항은 대통령령으로 정한다.'},
    {'id':'E04','category':'ETC', 'text':'제3조의 개정규정은 이 법 시행 후 최초로 접수된 사건부터 적용한다.'},
]

df = pd.DataFrame(sample_data)
df['length']      = df['text'].str.len()
df['token_count'] = df['text'].apply(lambda t: len(re.findall(r'[가-힣A-Za-z0-9]+', t)))

print('데이터셋: {} 건  /  {} 개 카테고리'.format(len(df), df['category'].nunique()))
print()
print('카테고리별 통계')
print(df.groupby('category')[['length', 'token_count']].agg(['count', 'mean']).round(1))


In [ ]:
# 5-2. 카테고리 분포 시각화

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['category'].value_counts().sort_index()
counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Category Sample Count')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].grid(axis='y', alpha=0.3)

lengths = df.groupby('category')['length'].mean().sort_index()
lengths.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='black')
axes[1].set_title('Average Text Length by Category')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Avg Characters')
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('균등 분포(각 4건)이므로 라벨 불균형 문제는 없습니다.')
print('실제 데이터에서는 특정 카테고리가 더 많을 수 있으므로 반드시 확인해야 합니다.')


## 6. 키워드 스코어링

규칙 기반 분류기를 만들기 전 단계로, 각 문장에서 카테고리별 키워드 점수를 계산합니다.

- score: 해당 카테고리 키워드가 몇 개 매칭됐는지
- top_category: 점수 1위 카테고리
- runner_up: 점수 2위 카테고리
- margin: 1위와 2위의 점수 차이. 작을수록 모호한 문장


In [ ]:
# 6-1. 카테고리 스코어링 + 모호도 분석

def score_by_category(text, rules):
    """카테고리별 키워드 매칭 점수와 매칭된 키워드 목록을 반환합니다."""
    score   = {cat: sum(1 for kw in kws if kw in text) for cat, kws in rules.items()}
    matched = {cat: [kw for kw in kws if kw in text]   for cat, kws in rules.items()}
    return score, matched


records = []
for _, row in df.iterrows():
    score, matched = score_by_category(row['text'], keyword_rules)
    sorted_cats    = sorted(score.items(), key=lambda x: x[1], reverse=True)
    top_cat, top_score   = sorted_cats[0]
    run_cat, run_score   = sorted_cats[1] if len(sorted_cats) > 1 else ('none', 0)
    records.append({
        'id':              row['id'],
        'category':        row['category'],
        'top_category':    top_cat,
        'top_score':       top_score,
        'runner_up':       run_cat,
        'margin':          top_score - run_score,
        'matched_keywords': {c: v for c, v in matched.items() if v},
        'text':            row['text'],
    })

scored_df = pd.DataFrame(records)


def ambiguity_label(row):
    if row['top_score'] == 0:
        return 'no_keyword'
    elif row['margin'] == 0:
        return 'tied'
    elif row['margin'] <= 1:
        return 'ambiguous'
    return 'confident'


scored_df['ambiguity'] = scored_df.apply(ambiguity_label, axis=1)

print('카테고리별 모호도 분포')
print(scored_df.groupby(['category', 'ambiguity']).size().unstack(fill_value=0))
print()
print('검토 우선순위가 높은 문장 (confident 아닌 것)')
ambiguous = scored_df[scored_df['ambiguity'] != 'confident']
print(ambiguous[['id', 'category', 'top_category', 'runner_up', 'margin', 'ambiguity', 'text']].to_string(index=False))


## 7. 규칙 기반 분류기 (Baseline)

동작 원리:
1. 각 카테고리의 키워드가 문장에 몇 개 있는지 셉니다.
2. 우선순위 순서대로 점수가 0보다 큰 첫 번째 카테고리를 반환합니다.
3. 아무 키워드도 없으면 ETC를 반환합니다.

이후 LLM 프롬프트 결과와 비교할 기준선이 됩니다.
단순 규칙도 일부 문장은 맞히지만, 문맥이 필요한 문장에서는 쉽게 실패합니다.


In [ ]:
# 7-1. 규칙 기반 분류기 구현 + 성능 평가

def rule_based_classify(text, priority=None):
    """우선순위 기반 규칙 분류기. 매칭 키워드가 없으면 ETC를 반환합니다."""
    priority = priority or ['DEF', 'ORG', 'CRIT', 'PROC', 'RIGHT', 'ETC']
    score, _ = score_by_category(text, keyword_rules)
    for cat in priority:
        if score[cat] > 0:
            return cat
    return 'ETC'


def evaluate_predictions(y_true, y_pred):
    """Precision, Recall, F1을 카테고리별로 계산합니다."""
    labels = sorted(set(y_true) | set(y_pred))
    rows = []
    for label in labels:
        tp = sum(t == label and p == label for t, p in zip(y_true, y_pred))
        fp = sum(t != label and p == label for t, p in zip(y_true, y_pred))
        fn = sum(t == label and p != label for t, p in zip(y_true, y_pred))
        pr = tp / (tp + fp) if tp + fp else 0
        rc = tp / (tp + fn) if tp + fn else 0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0
        rows.append({'category': label,
                     'precision': round(pr, 2),
                     'recall':    round(rc, 2),
                     'f1':        round(f1, 2),
                     'tp': tp, 'fp': fp, 'fn': fn})
    return pd.DataFrame(rows)


df['rule_pred'] = df['text'].apply(rule_based_classify)
df['correct']   = df['category'] == df['rule_pred']

acc = df['correct'].mean()
print('규칙 기반 분류기 정확도: {:.1%}'.format(acc))
print()
print('카테고리별 성능 지표')
metrics = evaluate_predictions(df['category'], df['rule_pred'])
print(metrics.to_string(index=False))
print()
print('혼동 행렬 (행=정답, 열=예측)')
confusion = pd.crosstab(df['category'], df['rule_pred'],
                        rownames=['정답'], colnames=['예측'])
print(confusion)


In [ ]:
# 7-2. 우선순위 조합 실험 — 성능 민감도 확인
#
# CRIT의 이하, 이내 같은 수치 표현이 처벌 조항(PROC)에도 등장합니다.
# 우선순위 순서 하나만 바꿔도 성능이 크게 달라지는 것을 확인합니다.

labels_no_etc = ['DEF', 'RIGHT', 'PROC', 'ORG', 'CRIT']
priority_results = []

for perm in itertools.permutations(labels_no_etc):
    priority = list(perm) + ['ETC']
    preds    = [rule_based_classify(t, priority) for t in df['text']]
    acc_val  = float(np.mean([yt == yp for yt, yp in zip(df['category'], preds)]))
    priority_results.append({'priority': ' > '.join(priority), 'accuracy': round(acc_val, 3)})

priority_df = pd.DataFrame(priority_results).sort_values('accuracy', ascending=False)

print('상위 5개 우선순위 조합')
print(priority_df.head(5).to_string(index=False))
print()
print('하위 5개 우선순위 조합')
print(priority_df.tail(5).to_string(index=False))
print()
print('정확도 범위: {:.1%} ~ {:.1%}'.format(
    priority_df['accuracy'].min(), priority_df['accuracy'].max()))
print('순서 하나만 바꿔도 성능이 크게 변합니다. 이것이 키워드 방식의 불안정성입니다.')


## 8. 애매한 실제형 문장 검증

두 카테고리 이상의 신호가 함께 나타나는 문장을 넣어
규칙 기반 분류기가 어떻게 실패하는지 확인합니다.

이런 문장들이 바로 LLM 프롬프트 엔지니어링이 필요한 이유입니다.


In [ ]:
# 8-1. 애매한 문장 검증

ambiguous_cases = pd.DataFrame([
    {
        'case_id':  'A01',
        'text':     '사업자는 안전관리 기준을 충족한 경우에만 서비스를 제공할 수 있다.',
        'expected': 'CRIT 또는 RIGHT',
        'note':     '조건이 핵심이면 CRIT, 사업자 의무가 핵심이면 RIGHT',
    },
    {
        'case_id':  'A02',
        'text':     '위원회는 신청인의 이의신청을 접수한 날부터 30일 이내에 심의하여야 한다.',
        'expected': 'ORG 또는 PROC',
        'note':     '위원회 권한 설명이면 ORG, 이의신청 처리 흐름이면 PROC',
    },
    {
        'case_id':  'A03',
        'text':     '세부 안전관리 기준은 대통령령으로 정한다.',
        'expected': 'CRIT 또는 ETC',
        'note':     '기준 내용 없이 위임만 있으면 ETC가 맞지만 키워드는 CRIT를 가리킴',
    },
    {
        'case_id':  'A04',
        'text':     '개인정보처리자는 정보주체의 열람 청구 절차를 지체 없이 안내하여야 한다.',
        'expected': 'RIGHT 또는 PROC',
        'note':     '정보주체 권리 보장이 핵심이면 RIGHT, 청구 절차가 핵심이면 PROC',
    },
])

ambiguous_cases['rule_pred']      = ambiguous_cases['text'].apply(rule_based_classify)
ambiguous_cases['matched_keywords'] = ambiguous_cases['text'].apply(
    lambda t: matched_categories(t, keyword_rules))

print('애매한 문장 — 규칙 기반 예측 vs 기대 라벨')
print()
for _, row in ambiguous_cases.iterrows():
    print('[{}] {}'.format(row['case_id'], row['text']))
    print('  기대 라벨  : {}'.format(row['expected']))
    print('  규칙 예측  : {}'.format(row['rule_pred']))
    print('  포인트     : {}'.format(row['note']))
    print('  매칭 키워드: {}'.format(row['matched_keywords']))
    print()

print('다음 강에서는 LLM에게 라벨 정의와 우선순위를 명시해서 이런 문장도 더 잘 처리합니다.')
